# TQ-IoT: Tabular Multi-Objective RL
## Battery Lifetime vs. Age of Information

This notebook implements a single-device IoT simulation with tabular multi-objective Q-learning.
It learns duty-cycling policies that trade off **energy consumption** and **Age of Information (AoI)** using **Chebyshev scalarisation**.

**State:** battery level, AoI, recent change in sensed value, channel quality

**Actions:** sleep | wait | transmit

**Rewards:** r_E = -cost(a),  r_F = -AoI(t)

## 1. Imports

In [ ]:
import numpy as np, random, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Tuple, List

## 2. Environment Configuration

In [ ]:
────────────────────────────────────────────────────────────────────
@dataclass
class EnvConfig:
    battery_cap: int = 100
    energy_cost_sleep: int = 1
    energy_cost_wait: int = 5
    energy_cost_transmit: int = 10
    aoi_max: int = 20          # cap AoI for binning
    aoi_bins: int = 5          # how many AoI bins
    battery_bins: int = 5      # how many battery bins
    horizon: int = 200         # max steps per episode
    # Channel quality: 0=poor, 1=medium, 2=good
    # Markov chain for channel quality
    channel_trans: np.ndarray = None

    def __post_init__(self):
        if self.channel_trans is None:
            self.channel_trans = np.array([
                [0.70, 0.25, 0.05],   # poor  -> poor/med/good
                [0.15, 0.70, 0.15],   # med   -> poor/med/good
                [0.05, 0.25, 0.70],   # good  -> poor/med/good
            ], dtype=float)

# ── Environment ───────────────────────────────────────────────────────────────
class TQIoTEnv:
    """
    State: (battery_bin, aoi_bin, delta_v_bin, channel_bin)
    Actions: 0=sleep, 1=wait(sense only), 2=transmit
    Rewards: r_E = -cost(a),  r_F = -AoI_t
    """
    ACTIONS = {0: "sleep", 1: "wait", 2: "transmit"}

    def __init__(self, cfg: EnvConfig = EnvConfig(), seed: int = 0):
        self.cfg = cfg
        self.rng = random.Random(seed)
        self.np_rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.battery = self.cfg.battery_cap
        self.aoi = 1                          # starts at 1
        self.sensed_value = self.np_rng.normal(0, 1)
        self.prev_sensed_value = self.sensed_value
        self.channel = 1                      # start medium
        self.t = 0
        return self._obs()

    def _battery_bin(self):
        b = self.cfg.battery_bins
        cap = self.cfg.battery_cap
        return min(b, max(0, int(self.battery / cap * b)))

    def _aoi_bin(self):
        return min(self.cfg.aoi_bins - 1,
                   int(self.aoi / self.cfg.aoi_max * self.cfg.aoi_bins))

    def _delta_v_bin(self):
        delta = abs(self.sensed_value - self.prev_sensed_value)
        # bin into 3: small / medium / large change
        if delta < 0.5:   return 0
        elif delta < 1.5: return 1
        else:             return 2

    def _obs(self):
        return (self._battery_bin(), self._aoi_bin(),
                self._delta_v_bin(), self.channel)

    def step(self, a: int):
        assert a in (0, 1, 2)
        costs = {0: self.cfg.energy_cost_sleep,
                 1: self.cfg.energy_cost_wait,
                 2: self.cfg.energy_cost_transmit}
        cost = costs[a]

        # battery depletion check
        if self.battery < cost:
            r_E = -float(self.battery)
            self.battery = 0
            return self._obs(), (r_E, -float(self.aoi)), True, {}

        self.battery -= cost

        # sense new value (always, even on sleep — but only transmit on a=2)
        self.prev_sensed_value = self.sensed_value
        self.sensed_value = self.np_rng.normal(self.sensed_value, 0.3)

        # AoI update: resets to 1 only on successful transmit
        if a == 2:
            # transmission success depends on channel quality
            tx_success_prob = [0.5, 0.8, 0.95][self.channel]
            if self.rng.random() < tx_success_prob:
                self.aoi = 1          # successful delivery — reset AoI
            else:
                self.aoi += 1         # failed tx — AoI still grows
        else:
            self.aoi += 1             # sleep or wait — AoI grows

        # cap AoI
        self.aoi = min(self.aoi, self.cfg.aoi_max)

        # rewards
        r_E = -float(cost)
        r_F = -float(self.aoi)        # r_F = -Delta(t)

        # evolve channel quality
        self.channel = int(np.argmax(
            self.np_rng.multinomial(1, self.cfg.channel_trans[self.channel])))

        self.t += 1
        done = (self.t >= self.cfg.horizon) or (self.battery <= 0)
        return self._obs(), (r_E, r_F), done, {"aoi": self.aoi, "battery": self.battery}

# ── State indexing ────────────────────────────────────────────────────────────
def num_states(cfg: EnvConfig = EnvConfig()) -> int:
    # battery_bins+1 * aoi_bins * 3 (delta_v) * 3 (channel)
    return (cfg.battery_bins + 1) * cfg.aoi_bins * 3 * 3

def state_to_index(s: Tuple, cfg: EnvConfig = EnvConfig()) -> int:
    b_bin, aoi_bin, dv_bin, ch_bin = s
    idx = b_bin
    idx = idx * cfg.aoi_bins + aoi_bin
    idx = idx * 3 + dv_bin
    idx = idx * 3 + ch_bin
    return idx

## 3. Agent: Two Q-Tables + Chebyshev Action Selection

In [ ]:
────────────────────────────────────────────────────────────
def num_states(cfg: EnvConfig = EnvConfig()) -> int:
    # battery_bins+1 * aoi_bins * 3 (delta_v) * 3 (channel)
    return (cfg.battery_bins + 1) * cfg.aoi_bins * 3 * 3

def state_to_index(s: Tuple, cfg: EnvConfig = EnvConfig()) -> int:
    b_bin, aoi_bin, dv_bin, ch_bin = s
    idx = b_bin
    idx = idx * cfg.aoi_bins + aoi_bin
    idx = idx * 3 + dv_bin
    idx = idx * 3 + ch_bin
    return idx

# ── Agent ─────────────────────────────────────────────────────────────────────
@dataclass
class AgentConfig:
    gamma: float = 0.95
    alpha: float = 0.2
    epsilon_start: float = 0.2
    epsilon_end: float = 0.05
    epsilon_decay_episodes: int = 300
    weights: List[Tuple[float, float]] = None

    def __post_init__(self):
        if self.weights is None:
            self.weights = [(round(w, 1), round(1.0 - w, 1))
                            for w in np.linspace(0.0, 1.0, 11)]

class TabularMORLAgent:
    def __init__(self, n_states, n_actions=3,
                 cfg: AgentConfig = AgentConfig(), seed: int = 0):
        self.cfg = cfg
        self.rng = random.Random(seed)
        self.QE = np.zeros((n_states, n_actions))   # energy Q-table
        self.QF = np.zeros((n_states, n_actions))   # AoI/freshness Q-table
        self.episode_count = 0
        self.n_actions = n_actions

    def epsilon(self):
        e0, e1, N = (self.cfg.epsilon_start, self.cfg.epsilon_end,
                     self.cfg.epsilon_decay_episodes)
        k = min(self.episode_count, N)
        return e0 + (e1 - e0) * (k / N)

    def chebyshev_scores(self, s, w):
        wE, wF = w
        zE = np.max(self.QE[s])
        zF = np.max(self.QF[s])
        dE = zE - self.QE[s]
        dF = zF - self.QF[s]
        return -np.maximum(wE * dE, wF * dF)

    def select_action(self, s, w):
        if self.rng.random() < self.epsilon():
            return self.rng.randrange(self.n_actions)
        scores = self.chebyshev_scores(s, w)
        best = np.flatnonzero(scores == np.max(scores))
        return int(self.rng.choice(list(best)))

    def update(self, s, a, rE, rF, s_next):
        g, alpha = self.cfg.gamma, self.cfg.alpha
        self.QE[s, a] += alpha * (rE + g * np.max(self.QE[s_next]) - self.QE[s, a])
        self.QF[s, a] += alpha * (rF + g * np.max(self.QF[s_next]) - self.QF[s, a])

    def train_for_weight(self, env, w, episodes=1, env_cfg=None):
        returns = []
        cfg = env_cfg or EnvConfig()
        for _ in range(episodes):
            self.episode_count += 1
            s = state_to_index(env.reset(), cfg)
            done, ep_ret = False, 0.0
            while not done:
                a = self.select_action(s, w)
                s_next, (rE, rF), done, _ = env.step(a)
                s_next = state_to_index(s_next, cfg)
                self.update(s, a, rE, rF, s_next)
                s = s_next
                ep_ret += w[0] * (-rE) + w[1] * (-rF)
            returns.append(ep_ret)
        return returns

## 4. Baseline Policies & Evaluation

In [ ]:
─────────────────────────────────────────────────────────────────
def evaluate_policy(agent, env, w, episodes=50, env_cfg=None):
    cfg = env_cfg or EnvConfig()
    energy_list, aoi_list, steps_list, peak_aoi_list, tx_list = [], [], [], [], []
    for _ in range(episodes):
        s = state_to_index(env.reset(), cfg)
        done = False
        e_sum, aoi_sum, steps, peak_aoi, tx_count = 0.0, 0.0, 0, 0, 0
        while not done:
            a = int(np.argmax(agent.chebyshev_scores(s, w)))
            s_next, (rE, rF), done, info = env.step(a)
            s_next = state_to_index(s_next, cfg)
            e_sum   += -rE
            aoi_sum += info.get("aoi", 0)
            peak_aoi = max(peak_aoi, info.get("aoi", 0))
            if a == 2: tx_count += 1
            steps += 1
            s = s_next
        energy_list.append(e_sum)
        aoi_list.append(aoi_sum / max(steps, 1))
        steps_list.append(steps)
        peak_aoi_list.append(peak_aoi)
        tx_list.append(tx_count / max(steps, 1))
    return {
        "avg_energy":    np.mean(energy_list),
        "mean_aoi":      np.mean(aoi_list),
        "peak_aoi":      np.mean(peak_aoi_list),
        "avg_steps":     np.mean(steps_list),
        "tx_rate":       np.mean(tx_list),
        "aoi_per_energy": np.mean(aoi_list) / max(np.mean(energy_list), 1e-6),
    }

def evaluate_baseline(env_cfg, action_fn, episodes=50):
    cfg = env_cfg
    energy_list, aoi_list, steps_list, peak_aoi_list, tx_list = [], [], [], [], []
    for ep in range(episodes):
        env = TQIoTEnv(cfg, seed=200 + ep)
        env.reset()
        done = False
        e_sum, aoi_sum, steps, peak_aoi, tx_count = 0.0, 0.0, 0, 0, 0
        while not done:
            a = action_fn(env._obs())
            _, (rE, rF), done, info = env.step(a)
            e_sum   += -rE
            aoi_sum += info.get("aoi", 0)
            peak_aoi = max(peak_aoi, info.get("aoi", 0))
            if a == 2: tx_count += 1
            steps += 1
        energy_list.append(e_sum)
        aoi_list.append(aoi_sum / max(steps, 1))
        steps_list.append(steps)
        peak_aoi_list.append(peak_aoi)
        tx_list.append(tx_count / max(steps, 1))
    return {
        "avg_energy":    np.mean(energy_list),
        "mean_aoi":      np.mean(aoi_list),
        "peak_aoi":      np.mean(peak_aoi_list),
        "avg_steps":     np.mean(steps_list),
        "tx_rate":       np.mean(tx_list),
        "aoi_per_energy": np.mean(aoi_list) / max(np.mean(energy_list), 1e-6),
    }

## 5. Pareto Mask

In [ ]:
───────────────────────────────────────────────────────────────
def compute_pareto_mask(df):
    # minimise both mean_aoi (lower=better) and avg_energy (lower=better)
    # maximise avg_steps (longer lifetime = better)
    mask = []
    for i, ri in df.iterrows():
        dominated = False
        for j, rj in df.iterrows():
            if i == j: continue
            # rj dominates ri if better on lifetime AND aoi
            if (rj["avg_steps"] >= ri["avg_steps"] and
                rj["mean_aoi"]  <= ri["mean_aoi"] and
                (rj["avg_steps"] > ri["avg_steps"] or rj["mean_aoi"] < ri["mean_aoi"])):
                dominated = True
                break
        mask.append(not dominated)
    return mask

## 6. Run Experiment

In [ ]:
───────────────────────────────────────────────────────────
def run_experiment(seed=42, train_eps_per_weight=100, eval_eps=50):
    env_cfg   = EnvConfig()
    n_states  = num_states(env_cfg)
    agent_cfg = AgentConfig()
    env_train = TQIoTEnv(env_cfg, seed=seed)
    agent     = TabularMORLAgent(n_states, cfg=agent_cfg, seed=seed)

    sample_efficiency = {w: [] for w in agent_cfg.weights}
    epsilons = []

    print("Training...")
    for _ in range(train_eps_per_weight):
        for w in agent_cfg.weights:
            ret = agent.train_for_weight(env_train, w, episodes=1, env_cfg=env_cfg)
            sample_efficiency[w].extend(ret)
            epsilons.append(agent.epsilon())

    print("Evaluating...")
    rows = []
    for w in agent_cfg.weights:
        env_eval = TQIoTEnv(env_cfg, seed=seed + 999)
        m = evaluate_policy(agent, env_eval, w, episodes=eval_eps, env_cfg=env_cfg)
        m["policy"] = f"TQ-IoT(wE={w[0]:.1f},wF={w[1]:.1f})"
        m["w_E"] = w[0]; m["w_F"] = w[1]
        rows.append(m)

    for label, fn in [("always_sleep",    lambda s: 0),
                      ("always_wait",     lambda s: 1),
                      ("always_transmit", lambda s: 2)]:
        m = evaluate_baseline(env_cfg, fn, episodes=eval_eps)
        m["policy"] = f"Baseline: {label}"
        m["w_E"] = None; m["w_F"] = None
        rows.append(m)

    df = pd.DataFrame(rows)
    df["pareto"] = compute_pareto_mask(df)
    return df, agent, epsilons, sample_efficiency, env_cfg

## 7. Train and Evaluate

In [ ]:
───────────────────────────────────────────────────────────────────────
df, agent, epsilons, sample_eff, env_cfg = run_experiment()
print(df[["policy","avg_steps","mean_aoi","peak_aoi","avg_energy","tx_rate"]].to_string(index=False))
df.to_csv("/home/claude/tq_iot_aoi_results.csv", index=False)
print("\nDone. Results saved.")

## 8. Results Table

In [ ]:
print(df[["policy","avg_steps","mean_aoi","peak_aoi","avg_energy","tx_rate"]].to_string(index=False))
df.to_csv("tq_iot_aoi_results.csv", index=False)

## 9. Pareto Front: Lifetime vs. Mean AoI

In [ ]:
─────────────────────────────────────────────────────────────────────
import os
os.makedirs("/home/claude/results", exist_ok=True)

learned  = df["policy"].str.startswith("TQ-IoT")
baseline = ~learned
df_l = df[learned].copy()
df_b = df[baseline].copy()

baseline_colors = {
    "Baseline: always_sleep":    "#e05a2b",
    "Baseline: always_wait":     "#f0a500",
    "Baseline: always_transmit": "#2ca05a",
}

## 10. Mean AoI vs. Preference Weight

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
is_pareto = df_l["pareto"].values
ax.scatter(df_l.loc[~is_pareto, "avg_steps"], df_l.loc[~is_pareto, "mean_aoi"],
           s=70, color="#1a5fa8", alpha=0.7, label="TQ-IoT (dominated)", zorder=3)
ax.scatter(df_l.loc[is_pareto, "avg_steps"], df_l.loc[is_pareto, "mean_aoi"],
           s=130, color="#1a5fa8", marker="^", label="TQ-IoT (Pareto-optimal)", zorder=4)
pareto_df = df_l[is_pareto].sort_values("avg_steps")
ax.plot(pareto_df["avg_steps"], pareto_df["mean_aoi"],
        "--", color="#1a5fa8", alpha=0.5, linewidth=1.5, label="Pareto front")
for _, row in df_b.iterrows():
    ax.scatter(row["avg_steps"], row["mean_aoi"], marker="x", s=200, linewidths=3,
               color=baseline_colors[row["policy"]], zorder=5,
               label=row["policy"].replace("Baseline: ", ""))
ax.set_xlabel("Device Lifetime (steps) — higher is better", fontsize=12)
ax.set_ylabel("Mean AoI — lower is better", fontsize=12)
ax.set_title("Multi-Objective Trade-off: Battery Lifetime vs. Age of Information", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/claude/results/pareto_lifetime_vs_aoi.png", dpi=180)
plt.close()

## 11. Device Lifetime vs. Preference Weight

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_l["w_E"], df_l["mean_aoi"], "o-", linewidth=2, color="#1a5fa8", label="TQ-IoT")
for _, row in df_b.iterrows():
    ax.axhline(row["mean_aoi"], linestyle="--", linewidth=1.8, alpha=0.7,
               color=baseline_colors[row["policy"]],
               label=row["policy"].replace("Baseline: ", ""))
ax.set_xlabel("Energy Weight (wE)", fontsize=12)
ax.set_ylabel("Mean AoI", fontsize=12)
ax.set_title("Mean Age of Information vs. Preference Weight", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/claude/results/mean_aoi_vs_weight.png", dpi=180)
plt.close()

## 12. Peak AoI vs. Preference Weight

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_l["w_E"], df_l["avg_steps"], "o-", linewidth=2, color="#1a5fa8", label="TQ-IoT")
for _, row in df_b.iterrows():
    ax.axhline(row["avg_steps"], linestyle="--", linewidth=1.8, alpha=0.7,
               color=baseline_colors[row["policy"]],
               label=row["policy"].replace("Baseline: ", ""))
ax.set_xlabel("Energy Weight (wE)", fontsize=12)
ax.set_ylabel("Device Lifetime (steps)", fontsize=12)
ax.set_title("Device Lifetime vs. Preference Weight", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/claude/results/lifetime_vs_weight.png", dpi=180)
plt.close()

## 13. Transmission Rate vs. Preference Weight

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_l["w_E"], df_l["peak_aoi"], "o-", linewidth=2, color="#c0392b", label="TQ-IoT")
for _, row in df_b.iterrows():
    ax.axhline(row["peak_aoi"], linestyle="--", linewidth=1.8, alpha=0.7,
               color=baseline_colors[row["policy"]],
               label=row["policy"].replace("Baseline: ", ""))
ax.set_xlabel("Energy Weight (wE)", fontsize=12)
ax.set_ylabel("Peak AoI", fontsize=12)
ax.set_title("Peak Age of Information vs. Preference Weight", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/claude/results/peak_aoi_vs_weight.png", dpi=180)
plt.close()

## 14. Exploration Decay

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_l["w_E"], df_l["tx_rate"] * 100, "o-", linewidth=2, color="#2ca05a", label="TQ-IoT")
ax.set_xlabel("Energy Weight (wE)", fontsize=12)
ax.set_ylabel("Transmission Rate (%)", fontsize=12)
ax.set_title("Transmission Frequency vs. Preference Weight", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/claude/results/tx_rate_vs_weight.png", dpi=180)
plt.close()

## 15. Cumulative Regret

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(epsilons)+1), epsilons, linewidth=1.5, color="#555")
ax.set_xlabel("Training Episodes", fontsize=12)
ax.set_ylabel("Exploration Rate (ε)", fontsize=12)
ax.set_title("Exploration Decay over Training", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/claude/results/epsilon_decay.png", dpi=180)
plt.close()

## 16. Sample Efficiency: Cumulative Returns

In [ ]:
agent_cfg = AgentConfig()
rep_weights = [agent_cfg.weights[0], agent_cfg.weights[3],
               agent_cfg.weights[5], agent_cfg.weights[7], agent_cfg.weights[-1]]
rep_colors  = ["#e05a2b","#f0a500","#888","#1a9e6f","#1a5fa8"]
rep_labels  = ["Quality-dominant (wE=0.0)","Quality-leaning (wE=0.3)",
               "Balanced (wE=0.5)","Energy-leaning (wE=0.7)","Energy-dominant (wE=1.0)"]

fig, ax = plt.subplots(figsize=(10, 6))
for w, c, lbl in zip(rep_weights, rep_colors, rep_labels):
    ret = np.array(sample_eff[w])
    regret = np.cumsum(np.maximum.accumulate(ret) - ret)
    ax.plot(range(1, len(regret)+1), regret, linewidth=2, color=c, label=lbl, alpha=0.85)
ax.set_xlabel("Training Episodes", fontsize=12)
ax.set_ylabel("Cumulative Regret", fontsize=12)
ax.set_title("Cumulative Training Regret for Representative Policies", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/home/claude/results/cumulative_regret.png", dpi=180)
plt.close()